In [ ]:
import os, sys, json, re, glob
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from tqdm.notebook import tqdm

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install spikingjelly rwkv deepspeed ninja fissix tomli
!pip install -r "/content/drive/MyDrive/UIT/HK6/CS338-NhanDang/SpikeGPT/requirements.txt"
# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
SPIKEGPT_REPO_DIR = '/content/drive/MyDrive/UIT/HK6/CS338-NhanDang/SpikeGPT'
MODEL_PATH = '/content/drive/MyDrive/UIT/HK6/CS338-NhanDang/SpikeGPT/Scratch78(best).pth'
DATA_PATH = '/content/drive/MyDrive/UIT/HK6/CS338-NhanDang/data/test_ood_data.jsonl'

HEAD_QK_DIM = 256
NUM_SAMPLES = 5000
# Sử dụng đường dẫn tuyệt đối để tránh lỗi khi đổi thư mục làm việc
OUTPUT_DIR = os.path.join(SPIKEGPT_REPO_DIR, 'eval_results')

model_name = os.path.basename(MODEL_PATH).replace('.pth', '')
run_dir = os.path.join(OUTPUT_DIR, f"{model_name}_HeadQK{HEAD_QK_DIM}")

if not os.path.exists(run_dir):
    os.makedirs(run_dir, exist_ok=True)
    print(f"Đã tạo thư mục: {run_dir}")
else:
    print(f"Thư mục đã tồn tại: {run_dir}")

In [ ]:
import os
os.chdir(SPIKEGPT_REPO_DIR)
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Set tham số môi trường TRƯỚC khi import model
os.environ["RWKV_HEAD_QK_DIM"] = str(HEAD_QK_DIM)
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_RUN_DEVICE"] = "cuda"

if SPIKEGPT_REPO_DIR not in sys.path:
    sys.path.append(SPIKEGPT_REPO_DIR)

from src.model import GPT, GPTConfig
from src.utils import TOKENIZER

# Khởi tạo mô hình
vocab_size = 50277
ctx_len = 1024
config = GPTConfig(vocab_size=vocab_size, ctx_len=ctx_len, model_type='RWKV', n_layer=18, n_embd=768)
model = GPT(config)

if os.path.exists(MODEL_PATH):
    w = torch.load(MODEL_PATH, map_location='cpu')
    load_result = model.load_state_dict(w, strict=False)
    print(f"Đã nạp tạ. Missing keys: {load_result.missing_keys}")
else:
    print(f"Lỗi: Không tìm thấy {MODEL_PATH}")

model = model.cuda()
model.eval()

# Khởi tạo Tokenizer
tokenizer_path = os.path.join(SPIKEGPT_REPO_DIR, "20B_tokenizer.json")
if not os.path.exists(tokenizer_path):
    # Fallback nếu đường dẫn hiện tại
    tokenizer_path = "20B_tokenizer.json"
tokenizer = TOKENIZER([tokenizer_path, tokenizer_path], UNKNOWN_CHAR=None)

def parse_dataset(filepath, num_samples=None):
    samples = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            data = json.loads(line)
            text = data['text']
            user_match = re.search(r"<\|im_start\|>user\n(.*?)<\|im_end\|>", text, re.DOTALL)
            user_prompt = user_match.group(1).strip() if user_match else ""
            gt_match = re.search(r"<tool_call>\n(.*?)\n</tool_call>", text, re.DOTALL)
            gt_json_str = gt_match.group(1).strip() if gt_match else "{}"
            try:
                gt_json = json.loads(gt_json_str)
            except:
                gt_json = {}
            if user_prompt and gt_json:
                samples.append({"prompt": user_prompt, "gt_name": gt_json.get("name", "unknown"), "gt_args": gt_json.get("arguments", {})})
            if num_samples and len(samples) >= num_samples:
                break
    return samples

def generate_text(model, tokenizer, prompt, max_new_tokens=150, ctx_len=1024):
    ctx = tokenizer.tokenizer.encode(prompt)
    src_len = len(ctx)
    for j in range(src_len):
        x = ctx[: j + 1]
        if j == src_len - 1:
            out = model.forward(torch.tensor([x], dtype=torch.long).cuda())
    out_tokens = []
    for k in range(src_len, src_len + max_new_tokens):
        x = ctx[: k + 1]
        x = x[-ctx_len:]
        out = model.forward(torch.tensor([x], dtype=torch.long).cuda())
        logits = out[0, -1]
        token = torch.argmax(logits).item()
        if token == 0: break
        out_tokens.append(token)
        ctx.append(token)
        if "</tool_call>" in tokenizer.tokenizer.decode(out_tokens):
            break
    return tokenizer.tokenizer.decode(out_tokens)

def extract_json_from_output(output_text):
    match = re.search(r"<tool_call>\n?(.*?)\n?</tool_call>", output_text, re.DOTALL)
    if not match:
        match = re.search(r"(\{.*?\})", output_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1).strip()), True
        except:
            return {"name": "invalid_json", "arguments": {}}, False
    return {"name": "no_json_found", "arguments": {}}, False

In [ ]:
samples = parse_dataset(DATA_PATH, num_samples=NUM_SAMPLES)
print(f"Tổng số mẫu test: {len(samples)}")

y_true_intent = []
y_pred_intent = []
metrics = {"valid_json_count": 0, "exact_args_match": 0}
results_log = []

print("Đang chạy suy luận...")
for i, sample in enumerate(tqdm(samples)):
    prompt = f"<|im_start|>system\nHãy thực hiện theo yêu cầu<|im_end|>\n<|im_start|>user\n{sample['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    with torch.no_grad():
        output_text = generate_text(model, tokenizer, prompt)

    pred_json, is_valid = extract_json_from_output(output_text)

    gt_name = sample['gt_name']
    pred_name = pred_json.get("name", "unknown")

    y_true_intent.append(gt_name)
    y_pred_intent.append(pred_name)

    if is_valid: metrics["valid_json_count"] += 1

    gt_args = sample['gt_args']
    pred_args = pred_json.get("arguments", {})
    is_args_exact = (gt_args == pred_args)
    if is_args_exact: metrics["exact_args_match"] += 1

    results_log.append({
        "prompt": sample['prompt'],
        "ground_truth": {"name": gt_name, "arguments": gt_args},
        "prediction": {"name": pred_name, "arguments": pred_args},
        "raw_output": output_text,
        "is_valid_json": is_valid,
        "is_args_exact": is_args_exact
    })

# Lưu Log để xem lại các câu sai
with open(os.path.join(run_dir, "predictions.json"), "w", encoding="utf-8") as f:
    json.dump(results_log, f, ensure_ascii=False, indent=2)
print("Đã lưu lịch sử suy luận.")

In [ ]:
valid_json_rate = metrics["valid_json_count"] / len(samples)
args_exact_rate = metrics["exact_args_match"] / len(samples)
intent_acc = accuracy_score(y_true_intent, y_pred_intent)
precision, recall, f1, _ = precision_recall_fscore_support(y_true_intent, y_pred_intent, average='weighted', zero_division=0)
report = classification_report(y_true_intent, y_pred_intent, zero_division=0)

report_text = f"=== BÁO CÁO ĐÁNH GIÁ MÔ HÌNH: {model_name} ===\n"
report_text += f"Cấu hình HeadQK: {HEAD_QK_DIM}\n"
report_text += "-"*40 + "\n"
report_text += f"1. Sinh JSON hợp lệ (Valid JSON) : {valid_json_rate:.2%}\n"
report_text += f"2. Chọn đúng Tool (Intent Acc)   : {intent_acc:.2%}\n"
report_text += f"3. Copy đúng ID (Args Exact Match): {args_exact_rate:.2%}\n"
report_text += "-"*40 + "\n"
report_text += f"Precision (Weighted): {precision:.4f}\n"
report_text += f"Recall (Weighted)   : {recall:.4f}\n"
report_text += f"F1 Score (Weighted) : {f1:.4f}\n"
report_text += "-"*40 + "\n"
report_text += "CHI TIẾT PHÂN LOẠI TOOL:\n" + report

with open(os.path.join(run_dir, "metrics_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)

print(report_text)

In [ ]:
# 1. Bar Chart Tổng quan
metrics_names = ['Valid JSON', 'Intent Accuracy', 'Args Exact Match', 'F1 Score']
metrics_values = [valid_json_rate, intent_acc, args_exact_rate, f1]

plt.figure(figsize=(10, 6))
sns.barplot(x=metrics_names, y=metrics_values, palette="viridis")
plt.ylim(0, 1.1)
plt.title(f"Hiệu suất Mô hình: {model_name} (HeadQK={HEAD_QK_DIM})", fontsize=14)
for i, v in enumerate(metrics_values):
    plt.text(i, v + 0.02, f"{v:.1%}", ha='center', fontsize=12)
plt.ylabel('Tỷ lệ', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(run_dir, "overall_performance.png"), dpi=300)
plt.show()

# 2. Confusion Matrix
labels = sorted(list(set(y_true_intent + y_pred_intent)))
cm = confusion_matrix(y_true_intent, y_pred_intent, labels=labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix - Chọn Tool (Intent)", fontsize=14)
plt.ylabel("Thực tế (Ground Truth)")
plt.xlabel("Mô hình dự đoán (Prediction)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(run_dir, "intent_confusion_matrix.png"), dpi=300)
plt.show()

print(f"\nĐánh giá hoàn tất! File Report và Biểu đồ đã được lưu vào thư mục: {run_dir}/")